## Step 1

Hitting endpoint: https://catalog.redhat.com/api/containers/v1/repositories/registry/registry.access.redhat.com/repository/rhbk%2Fkeycloak-rhel9/images

### Return:

a json with list of all the image metadata and data in the specified Repository (rhbk/keycloak-rhel9)

#### Next Step:

The next step will be analyzing the retrieved images to identify key fields that can be used to group images by their content stream

In [ ]:
import requests
from datetime import datetime

REDHAT_KEYCLOAK_IMAGES_URL = "https://catalog.redhat.com/api/containers/v1/repositories/registry/registry.access.redhat.com/repository/rhbk%2Fkeycloak-rhel9/images"


def fetch_keycloak_images_in_catalog():
    request_headers = {"accept": "application/json"}
    request_response = requests.get(REDHAT_KEYCLOAK_IMAGES_URL, headers=request_headers)
    request_response.raise_for_status()
    return request_response.json()


## Step 2

Filter the returned object from fetch_keycloak_images_in_catalog() function as per Content streams

### Return

the same list passed as argument but grouped according to the content stream found under label->version->value

#### Next

in each content stream find the recent image by comparing fields like published_date, last_update_date, creation_date and return only 1 image(recent) per content stream

In [8]:
def group_images_by_version_label(api_response_data):
    images_grouped_by_version = {}

    for image_entry in api_response_data.get("data", []):
        parsed_data_labels = image_entry.get("parsed_data", {}).get("labels", [])
        version_label_value = None

        for label_entry in parsed_data_labels:
            if label_entry.get("name") == "version":
                version_label_value = label_entry.get("value")
                break

        if version_label_value:
            if version_label_value not in images_grouped_by_version:
                images_grouped_by_version[version_label_value] = []
            images_grouped_by_version[version_label_value].append(image_entry)

    return images_grouped_by_version

In [ ]:
sample = fetch_keycloak_images_in_catalog()
group_images_by_version_label(sample)